# 03/04 — Knowledge & Citation Edges (2022–2025)

**Combines two steps:**
- Clean and finalise `knowledge_edges_2022_2025.csv`
- Build and save `citation_edges_2022_2025.csv`

**Predicate cleaning rationale:**
- The triplet extraction model produces single-token predicates, many of which are punctuation, stopwords, or numbers
- These are removed as they carry no semantic meaning for TransE relation embeddings
- This is a known limitation of automated KG construction from scientific text — documented in decision log

In [16]:
import pandas as pd
import ast
import os

# Paths 
OUT_DIR   = "../outputs/final/"
OTHER_DIR = "../outputs/other/"

print("Setup complete.")

Setup complete.


## Part 1 — Clean Knowledge Edges

In [17]:
ke = pd.read_csv(OUT_DIR + 'knowledge_edges_2022_2025.csv')
print(f"Loaded: {len(ke):,} edges, {ke['predicate'].nunique():,} unique predicates")
print(ke['predicate'].value_counts().head(10))

Loaded: 292,342 edges, 8,706 unique predicates
predicate
example     3005
also        2026
addition    1817
learning    1815
study       1717
authors     1577
use         1483
data        1480
research    1377
first       1258
Name: count, dtype: int64


In [25]:
# Predicate noise filter
STOPWORDS = {
    'the','a','an','is','are','was','were','be','been','being',
    'and','or','but','in','of','to','for','on','at','by','with',
    'from','this','that','these','those','it','its','we','our',
    'they','them','their','i','my','he','she','his','her',
    'not','no','nor','so','yet','both','either','neither',
    'has','have','had','do','does','did','will','would','could',
    'should','may','might','shall','can','need','dare','used',
    'et','al','vs','via','per','eg','ie', ',',
    'also','then','than','when','where','while','which','who',
    'what','how','why','all','any','each','few','more','most',
    'other','such','same','own','just','now','here','there',
    'very','too','quite','rather','some','many','much','well',
}

def is_noise(pred):
    """Returns True if predicate should be dropped."""
    if pd.isna(pred): return True
    s = str(pred).strip()
    if len(s) == 0: return True
    if not any(c.isalpha() for c in s): return True   # pure punctuation/numbers
    if len(s) == 1: return True                        # single character
    if s.lower() in STOPWORDS: return True             # stopword
    return False

# Step 1: noise removal
mask_noise = ke['predicate'].apply(is_noise)
clean = ke[~mask_noise].reset_index(drop=True)
print(f"After noise removal : {len(clean):,} edges  (removed {mask_noise.sum():,})")

# Step 2: frequency cutoff — keep predicates appearing ≥5 times
freq = clean['predicate'].value_counts()
valid_preds = freq[freq >= 5].index
final_ke = clean[clean['predicate'].isin(valid_preds)].reset_index(drop=True)
print(f"After freq >= 5     : {len(final_ke):,} edges, {final_ke['predicate'].nunique():,} unique predicates")
print()
print(f"Removed {len(ke) - len(final_ke):,} edges total ({(len(ke)-len(final_ke))/len(ke)*100:.1f}%)")

After noise removal : 290,316 edges  (removed 2,026)
After freq >= 5     : 290,316 edges, 8,705 unique predicates

Removed 2,026 edges total (0.7%)


In [26]:
# Verify split breakdown
final_ke['split'] = final_ke['source'].apply(lambda x: x.split('_')[0])
print("Edges per split:")
print(final_ke['split'].value_counts().to_string())
final_ke = final_ke.drop(columns=['split'])
print()
print(f"Unique papers with edges : {final_ke['source'].nunique():,}")
print(f"Year range               : {final_ke['year'].min()} – {final_ke['year'].max()}")
print(f"Missing years            : {final_ke['year'].isna().sum()}")
print()
print("Top 20 predicates:")
print(final_ke['predicate'].value_counts().head(20).to_string())

Edges per split:
split
SKG      144179
BLOG     115898
NOVEL     30239

Unique papers with edges : 873
Year range               : 2022 – 2025
Missing years            : 0

Top 20 predicates:
predicate
example     3005
addition    1817
learning    1815
study       1717
authors     1577
use         1483
data        1480
research    1377
first       1258
results     1159
models      1129
review      1107
language    1068
studies     1068
instance    1029
model        974
AI           939
role         938
work         937
analysis     868


In [27]:
# Save cleaned knowledge edges
final_ke = final_ke[['source', 'target', 'predicate', 'year']]
final_ke.to_csv(OUT_DIR + 'knowledge_edges_2022_2025.csv', index=False)
print(f"Saved: knowledge_edges_2022_2025.csv  ({len(final_ke):,} edges)")

Saved: knowledge_edges_2022_2025.csv  (290,316 edges)


## Part 2 — Citation Edges

In [28]:
meta   = pd.read_csv(OUT_DIR + 'openalex_metadata_2022_2025.csv')
papers = pd.read_csv(OUT_DIR + 'paper_nodes_2022_2025.csv')

print(f"Metadata rows : {len(meta):,}")
print(f"Paper nodes   : {len(papers):,}")
print(f"No referenced_works: {meta['referenced_works'].isna().sum():,}")

Metadata rows : 2,331
Paper nodes   : 2,331
No referenced_works: 0


In [29]:
citation_edges = []

for _, row in meta.iterrows():
    source = row['global_paper_id']
    year   = row['year']

    if pd.isna(row['referenced_works']):
        continue

    try:
        refs = ast.literal_eval(row['referenced_works'])
    except Exception:
        continue

    for ref_url in refs:
        # Strip full OpenAlex URL to W-format ID for consistency
        # e.g. 'https://openalex.org/W1234567' → 'W1234567'
        ref_id = str(ref_url).strip().rstrip('/').split('/')[-1]
        citation_edges.append({
            'source' : source,
            'target' : ref_id,
            'year'   : year
        })

citation_df = pd.DataFrame(citation_edges)
print(f"Raw citation edges     : {len(citation_df):,}")

# Deduplicate
citation_df = citation_df.drop_duplicates().reset_index(drop=True)
print(f"After dedup            : {len(citation_df):,}")
print(f"Papers with citations  : {citation_df['source'].nunique():,}")
print()
print("Sample targets (should be W-format):")
print(citation_df['target'].head(5).tolist())

Raw citation edges     : 320,724
After dedup            : 320,724
Papers with citations  : 2,235

Sample targets (should be W-format):
['W601952952', 'W648786980', 'W1502957213', 'W1524281572', 'W1591706642']


In [30]:
# Coverage check
papers_with_ke   = set(final_ke['source'])
papers_with_cite = set(citation_df['source'])
all_paper_ids    = set(papers['node_id'])

covered = papers_with_ke | papers_with_cite
isolated = all_paper_ids - covered

print(f"Papers with knowledge edges only : {len(papers_with_ke - papers_with_cite):,}")
print(f"Papers with citation edges only  : {len(papers_with_cite - papers_with_ke):,}")
print(f"Papers with both                 : {len(papers_with_ke & papers_with_cite):,}")
print(f"Fully isolated (no edges at all) : {len(isolated):,}")
print(f"Total papers                     : {len(all_paper_ids):,}")

Papers with knowledge edges only : 38
Papers with citation edges only  : 1,400
Papers with both                 : 835
Fully isolated (no edges at all) : 58
Total papers                     : 2,331


In [31]:
# Save
citation_df.to_csv(OTHER_DIR + 'citation_edges_2022_2025.csv', index=False)
print(f"Saved: citation_edges_2022_2025.csv  ({len(citation_df):,} edges)")
print(f"knowledge_edges_2022_2025.csv : {len(final_ke):,} edges")
print(f"citation_edges_2022_2025.csv  : {len(citation_df):,} edges")
print(f"Fully isolated papers         : {len(isolated):,}")

Saved: citation_edges_2022_2025.csv  (320,724 edges)
knowledge_edges_2022_2025.csv : 290,316 edges
citation_edges_2022_2025.csv  : 320,724 edges
Fully isolated papers         : 58
